# Evaluation of precomputed full-time-series archives

This notebook reads existing reference, standard M3C2, and TAM3C2 archives and evaluates their stored 4D-OBC results. It does not rerun distance estimation or object extraction.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import glob
import os

import numpy as np
import matplotlib.pyplot as plt

import py4dgeo


## 1. Configuration

In [ ]:
# Folder created by the scenario-specific producer notebook.
# It may be an absolute path or a folder relative to this notebook.
archive_folder = os.path.join(
    os.getcwd(),
    "simulation_test2_als_downsampled1",
)

# Each specification may be an exact filename, an absolute ZIP path, or a glob
# pattern. A glob pattern must resolve to exactly one archive.
# Normal and spatial-gap producer notebooks normally use these two patterns.
tam3c2_archive_spec = "*_full_timeseries_best_scale_idx*_weighted_tam3c2.zip"
m3c2_archive_spec = "*_full_timeseries_best_scale_idx*_standard_m3c2.zip"

# For the temporal-gap notebook that reconstructs withheld timestamps, use:
# tam3c2_archive_spec = "*_full_timeline_reconstructed_best_scale_idx*_weighted_tam3c2.zip"
# m3c2_archive_spec = "*_observed_epochs_only_best_scale_idx*_standard_m3c2.zip"

# The mesh-derived reference archive is independent of the selected scenario.
reference_archive_path = os.path.join(
    os.getcwd(),
    "simulation2_reference.zip",
)

# Object-level matching threshold used by the evaluation below.
match_iou_threshold = 0.80


## 2. Load precomputed analysis archives


In [ ]:
def resolve_archive(folder, archive_spec, label):
    folder = os.path.abspath(folder)
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Archive folder not found: {folder}")

    candidate = (
        archive_spec
        if os.path.isabs(archive_spec)
        else os.path.join(folder, archive_spec)
    )

    if glob.has_magic(candidate):
        matches = sorted(glob.glob(candidate))

        # If the specified folder is a parent directory, also search below it.
        if not matches and not os.path.isabs(archive_spec):
            recursive_pattern = os.path.join(folder, "**", archive_spec)
            matches = sorted(glob.glob(recursive_pattern, recursive=True))

        matches = [match for match in matches if os.path.isfile(match)]
        if len(matches) != 1:
            available_zip_files = sorted(
                glob.glob(os.path.join(folder, "**", "*.zip"), recursive=True)
            )
            available_text = "\n".join(
                f"  - {zip_path}" for zip_path in available_zip_files
            ) or "  (no ZIP files found)"
            raise ValueError(
                f"{label} archive pattern must match exactly one file; "
                f"found {len(matches)} matches for {archive_spec!r} under {folder!r}.\n"
                f"Available ZIP files:\n{available_text}"
            )
        candidate = matches[0]

    if not os.path.isfile(candidate):
        available_zip_files = sorted(
            glob.glob(os.path.join(folder, "**", "*.zip"), recursive=True)
        )
        available_text = "\n".join(
            f"  - {zip_path}" for zip_path in available_zip_files
        ) or "  (no ZIP files found)"
        raise FileNotFoundError(
            f"{label} archive not found: {candidate}\n"
            f"Available ZIP files under {folder}:\n{available_text}"
        )
    return os.path.abspath(candidate)

gt_path = os.path.abspath(reference_archive_path)
if not os.path.isfile(gt_path):
    raise FileNotFoundError(f"Reference archive not found: {gt_path}")

tam3c2_path = resolve_archive(
    archive_folder,
    tam3c2_archive_spec,
    "TAM3C2",
)
m3c2_path = resolve_archive(
    archive_folder,
    m3c2_archive_spec,
    "M3C2",
)

ref_analysis = py4dgeo.SpatiotemporalAnalysis(gt_path, force=False)
analysis = py4dgeo.SpatiotemporalAnalysis(tam3c2_path, force=False)
m3c2_analysis = py4dgeo.SpatiotemporalAnalysis(m3c2_path, force=False)

analyses = {
    "GT": ref_analysis,
    "M3C2": m3c2_analysis,
    "TAM3C2": analysis,
}

for name, st_analysis in analyses.items():
    if st_analysis.objects is None:
        raise RuntimeError(
            f"{name} archive contains no stored 4D-OBC objects: "
            f"{st_analysis.filename}. Run object extraction in the producer notebook first."
        )
    print(
        f"{name}: {len(st_analysis.objects)} objects, "
        f"{st_analysis.corepoints.cloud.shape[0]:,} corepoints, "
        f"distance shape={st_analysis.distances.shape}, "
        f"path={st_analysis.filename}"
    )


## 9. Compare GT, M3C2, and TAM3C2 4D-OBCs

The comparison treats every 4D-OBC as a Cartesian product of its core-point set and its continuous timestamp interval. Object matching is one-to-one and maximizes total spatiotemporal IoU before applying the match threshold.

In [ ]:
# The archives were loaded in Section 2. Keep the shared dictionary interface
# used by the evaluation code below.
analyses = {
    'GT': ref_analysis,
    'M3C2': m3c2_analysis,
    'TAM3C2': analysis,
}

for name, st_analysis in analyses.items():
    objects_for_method = st_analysis.objects
    if objects_for_method is None:
        raise RuntimeError(
            f"{name} analysis has no stored 4D-OBC objects. "
            "Run RegionGrowingAlgorithm in the producer notebook first."
        )
    print(
        f"{name}: {len(objects_for_method)} objects, "
        f"{st_analysis.corepoints.cloud.shape[0]:,} corepoints, "
        f"distance shape={st_analysis.distances.shape}, "
        f"path={st_analysis.filename}"
    )


In [ ]:
from scipy.optimize import linear_sum_assignment

try:
    import pandas as pd
except ImportError:
    pd = None


_IOU_TOL = 1e-12


def _timestamps_for_analysis(st_analysis):
    timedeltas = list(st_analysis.timedeltas)
    if len(timedeltas) != st_analysis.distances.shape[1]:
        raise ValueError(
            f"{st_analysis.filename}: {len(timedeltas)} timestamps for "
            f"{st_analysis.distances.shape[1]} distance epochs"
        )

    reference_time = st_analysis.reference_epoch.timestamp
    timestamps = np.array([reference_time + td for td in timedeltas], dtype=object)
    if len(timestamps) == 0:
        raise ValueError(f"{st_analysis.filename}: no acquisition timestamps available")

    time_days = np.array(
        [(ts - timestamps[0]).total_seconds() / 86400.0 for ts in timestamps],
        dtype=float,
    )
    return timestamps, time_days


def _validate_common_corepoints(analyses):
    base_name = 'GT'
    base_corepoints = np.asarray(analyses[base_name].corepoints.cloud)
    for name, st_analysis in analyses.items():
        corepoints_for_method = np.asarray(st_analysis.corepoints.cloud)
        if corepoints_for_method.shape != base_corepoints.shape or not np.allclose(
            corepoints_for_method,
            base_corepoints,
        ):
            raise ValueError(
                f"{name} corepoints differ from {base_name}; this evaluation assumes "
                "identical core-point arrays and indices."
            )
    print(
        f"All analyses use the same {base_corepoints.shape[0]:,} corepoints; "
        "nearest-neighbor corepoint mapping is bypassed."
    )


def _object_properties(obj, time_days):
    corepoints_for_object = set(int(idx) for idx in np.asarray(obj.indices, dtype=int))
    start_epoch = int(obj.start_epoch)
    end_epoch = int(obj.end_epoch)

    if start_epoch > end_epoch:
        raise ValueError(f"Invalid object interval: {start_epoch} > {end_epoch}")
    if start_epoch < 0 or end_epoch >= len(time_days):
        raise IndexError(
            f"Object epoch interval [{start_epoch}, {end_epoch}] outside "
            f"timestamp array of length {len(time_days)}"
        )
    if not corepoints_for_object:
        raise ValueError("Object has empty corepoint support")

    start_time = float(time_days[start_epoch])
    end_time = float(time_days[end_epoch])
    duration = end_time - start_time
    if duration < 0:
        raise ValueError(f"Object has negative duration: {duration}")

    return {
        'corepoints': corepoints_for_object,
        'start_epoch': start_epoch,
        'end_epoch': end_epoch,
        'start_time': start_time,
        'end_time': end_time,
        'duration': duration,
        'n_corepoints': len(corepoints_for_object),
    }


def _pairwise_object_metrics(det_obj, ref_obj, det_time_days, ref_time_days):
    det = _object_properties(det_obj, det_time_days)
    ref = _object_properties(ref_obj, ref_time_days)

    shared_corepoint_count = len(det['corepoints'] & ref['corepoints'])
    spatial_union_count = det['n_corepoints'] + ref['n_corepoints'] - shared_corepoint_count
    spatial_iou = (
        shared_corepoint_count / spatial_union_count
        if spatial_union_count > 0
        else np.nan
    )

    overlapping_duration = max(
        0.0,
        min(det['end_time'], ref['end_time'])
        - max(det['start_time'], ref['start_time']),
    )
    temporal_union = det['duration'] + ref['duration'] - overlapping_duration
    temporal_iou = overlapping_duration / temporal_union if temporal_union > 0 else np.nan

    detected_st_support = det['n_corepoints'] * det['duration']
    reference_st_support = ref['n_corepoints'] * ref['duration']
    intersection_st_support = shared_corepoint_count * overlapping_duration
    union_st_support = detected_st_support + reference_st_support - intersection_st_support
    spatiotemporal_iou = (
        intersection_st_support / union_st_support
        if union_st_support > 0
        else np.nan
    )

    if np.isfinite(spatiotemporal_iou):
        if np.isfinite(spatial_iou) and spatiotemporal_iou > spatial_iou + _IOU_TOL:
            raise AssertionError("Spatiotemporal IoU exceeds spatial IoU")
        if np.isfinite(temporal_iou) and spatiotemporal_iou > temporal_iou + _IOU_TOL:
            raise AssertionError("Spatiotemporal IoU exceeds temporal IoU")

    return {
        'detected_corepoint_count': det['n_corepoints'],
        'reference_corepoint_count': ref['n_corepoints'],
        'shared_corepoint_count': shared_corepoint_count,
        'detected_start_time': det['start_time'],
        'detected_end_time': det['end_time'],
        'reference_start_time': ref['start_time'],
        'reference_end_time': ref['end_time'],
        'detected_duration': det['duration'],
        'reference_duration': ref['duration'],
        'overlapping_duration': overlapping_duration,
        'spatial_iou': spatial_iou,
        'temporal_iou': temporal_iou,
        'spatiotemporal_iou': spatiotemporal_iou,
        'detected_spatiotemporal_support': detected_st_support,
        'reference_spatiotemporal_support': reference_st_support,
        'intersection_spatiotemporal_support': intersection_st_support,
        'union_spatiotemporal_support': union_st_support,
    }


def _pairwise_metrics_table(method, detected_objects, reference_objects, det_time_days, ref_time_days):
    rows = []
    for detected_idx, det_obj in enumerate(detected_objects):
        for reference_idx, ref_obj in enumerate(reference_objects):
            row = {
                'method': method,
                'detected_object_index': detected_idx,
                'reference_object_index': reference_idx,
            }
            row.update(_pairwise_object_metrics(det_obj, ref_obj, det_time_days, ref_time_days))
            rows.append(row)
    return rows


def _spatiotemporal_iou_matrix(pair_rows, n_detected, n_reference):
    matrix = np.full((n_detected, n_reference), np.nan, dtype=float)
    for row in pair_rows:
        matrix[row['detected_object_index'], row['reference_object_index']] = row[
            'spatiotemporal_iou'
        ]
    return matrix


def _one_to_one_matches(spatiotemporal_iou_matrix, threshold):
    if 0 in spatiotemporal_iou_matrix.shape:
        return []

    cost = np.where(
        np.isfinite(spatiotemporal_iou_matrix),
        1.0 - spatiotemporal_iou_matrix,
        1e6,
    )
    detected_indices, reference_indices = linear_sum_assignment(cost)
    matches = []
    for detected_idx, reference_idx in zip(detected_indices, reference_indices):
        iou = spatiotemporal_iou_matrix[detected_idx, reference_idx]
        if np.isfinite(iou) and iou >= threshold:
            matches.append((int(detected_idx), int(reference_idx)))
    return matches


def _merge_intervals(intervals):
    merged = []
    for start_time, end_time in sorted(intervals):
        if not np.isfinite(start_time) or not np.isfinite(end_time):
            continue
        if end_time < start_time:
            raise ValueError(f"Invalid interval [{start_time}, {end_time}]")
        if not merged or start_time > merged[-1][1] + _IOU_TOL:
            merged.append([float(start_time), float(end_time)])
        else:
            merged[-1][1] = max(merged[-1][1], float(end_time))
    return [(start_time, end_time) for start_time, end_time in merged]


def _intervals_by_corepoint(objects_for_method, time_days):
    intervals = {}
    for obj in objects_for_method:
        props = _object_properties(obj, time_days)
        if props['duration'] == 0:
            continue
        interval = (props['start_time'], props['end_time'])
        for corepoint_idx in props['corepoints']:
            intervals.setdefault(corepoint_idx, []).append(interval)
    return {
        corepoint_idx: _merge_intervals(corepoint_intervals)
        for corepoint_idx, corepoint_intervals in intervals.items()
    }


def _total_interval_duration(intervals):
    return float(sum(end_time - start_time for start_time, end_time in intervals))


def _shared_interval_duration(left_intervals, right_intervals):
    total = 0.0
    left_idx = 0
    right_idx = 0
    while left_idx < len(left_intervals) and right_idx < len(right_intervals):
        left_start, left_end = left_intervals[left_idx]
        right_start, right_end = right_intervals[right_idx]
        total += max(0.0, min(left_end, right_end) - max(left_start, right_start))
        if left_end <= right_end:
            left_idx += 1
        else:
            right_idx += 1
    return total


def _global_support_metrics(detected_objects, reference_objects, det_time_days, ref_time_days):
    detected_by_corepoint = _intervals_by_corepoint(detected_objects, det_time_days)
    reference_by_corepoint = _intervals_by_corepoint(reference_objects, ref_time_days)
    corepoint_indices = set(detected_by_corepoint) | set(reference_by_corepoint)

    tp = 0.0
    detected_total = 0.0
    reference_total = 0.0
    for corepoint_idx in corepoint_indices:
        detected_intervals = detected_by_corepoint.get(corepoint_idx, [])
        reference_intervals = reference_by_corepoint.get(corepoint_idx, [])
        detected_total += _total_interval_duration(detected_intervals)
        reference_total += _total_interval_duration(reference_intervals)
        tp += _shared_interval_duration(detected_intervals, reference_intervals)

    fp = max(0.0, detected_total - tp)
    fn = max(0.0, reference_total - tp)
    precision = tp / (tp + fp) if tp + fp > 0 else np.nan
    recall = tp / (tp + fn) if tp + fn > 0 else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else np.nan
    iou = tp / (tp + fp + fn) if tp + fp + fn > 0 else np.nan

    return {
        'global_tp': tp,
        'global_fp': fp,
        'global_fn': fn,
        'global_precision': precision,
        'global_recall': recall,
        'global_f1': f1,
        'global_iou': iou,
    }


def _trapezoid(values, times):
    if hasattr(np, 'trapezoid'):
        return float(np.trapezoid(values, times))
    return float(np.trapz(values, times))


def _mean_time_integrated_change(obj, distances, time_days):
    props = _object_properties(obj, time_days)
    epoch_slice = slice(props['start_epoch'], props['end_epoch'] + 1)
    object_times = time_days[epoch_slice] - time_days[props['start_epoch']]
    if len(object_times) < 2:
        return {
            'change_volume': np.nan,
            'valid_corepoint_count': 0,
            'valid_corepoint_percentage': 0.0,
        }

    integrated_changes = []
    for corepoint_idx in sorted(props['corepoints']):
        series = np.asarray(distances[corepoint_idx, epoch_slice], dtype=float)
        if not np.isfinite(series[0]):
            continue
        finite = np.isfinite(series)
        if finite.sum() < 2:
            continue
        change = np.abs(series[finite] - series[0])
        integrated_changes.append(_trapezoid(change, object_times[finite]))

    valid_count = len(integrated_changes)
    return {
        'change_volume': float(np.mean(integrated_changes)) if valid_count else np.nan,
        'valid_corepoint_count': valid_count,
        'valid_corepoint_percentage': 100.0 * valid_count / props['n_corepoints'],
    }


def _summary_stats(values, prefix):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {
            f'mean_{prefix}': np.nan,
            f'median_{prefix}': np.nan,
            f'std_{prefix}': np.nan,
            f'min_{prefix}': np.nan,
            f'max_{prefix}': np.nan,
        }
    return {
        f'mean_{prefix}': float(np.mean(values)),
        f'median_{prefix}': float(np.median(values)),
        f'std_{prefix}': float(np.std(values)),
        f'min_{prefix}': float(np.min(values)),
        f'max_{prefix}': float(np.max(values)),
    }


def _harmonic_mean(precision, recall):
    return 2 * precision * recall / (precision + recall) if precision + recall > 0 else np.nan


def _evaluate_method(method, method_analysis, gt_analysis, timestamp_info):
    detected_objects = list(method_analysis.objects)
    reference_objects = list(gt_analysis.objects)
    detected_timestamps, detected_time_days = timestamp_info[method]
    reference_timestamps, reference_time_days = timestamp_info['GT']

    pair_rows = _pairwise_metrics_table(
        method,
        detected_objects,
        reference_objects,
        detected_time_days,
        reference_time_days,
    )
    pair_lookup = {
        (row['detected_object_index'], row['reference_object_index']): row
        for row in pair_rows
    }
    iou_matrix = _spatiotemporal_iou_matrix(
        pair_rows,
        len(detected_objects),
        len(reference_objects),
    )
    matched_pairs = _one_to_one_matches(iou_matrix, match_iou_threshold)

    matched_rows = []
    for detected_idx, reference_idx in matched_pairs:
        pair_row = pair_lookup[(detected_idx, reference_idx)]
        detected_change = _mean_time_integrated_change(
            detected_objects[detected_idx],
            method_analysis.distances,
            detected_time_days,
        )
        reference_change = _mean_time_integrated_change(
            reference_objects[reference_idx],
            gt_analysis.distances,
            reference_time_days,
        )
        change_difference = detected_change['change_volume'] - reference_change['change_volume']
        absolute_change_difference = abs(change_difference)

        matched_rows.append({
            'method': method,
            'detected_object_index': detected_idx,
            'reference_object_index': reference_idx,
            'spatial_iou': pair_row['spatial_iou'],
            'temporal_iou': pair_row['temporal_iou'],
            'spatiotemporal_iou': pair_row['spatiotemporal_iou'],
            'detected_corepoint_count': pair_row['detected_corepoint_count'],
            'reference_corepoint_count': pair_row['reference_corepoint_count'],
            'shared_corepoint_count': pair_row['shared_corepoint_count'],
            'detected_start_timestamp': detected_timestamps[int(detected_objects[detected_idx].start_epoch)],
            'detected_end_timestamp': detected_timestamps[int(detected_objects[detected_idx].end_epoch)],
            'reference_start_timestamp': reference_timestamps[int(reference_objects[reference_idx].start_epoch)],
            'reference_end_timestamp': reference_timestamps[int(reference_objects[reference_idx].end_epoch)],
            'detected_duration_days': pair_row['detected_duration'],
            'reference_duration_days': pair_row['reference_duration'],
            'overlapping_duration_days': pair_row['overlapping_duration'],
            'detected_change_volume': detected_change['change_volume'],
            'reference_change_volume': reference_change['change_volume'],
            'change_volume_difference': change_difference,
            'absolute_change_volume_difference': absolute_change_difference,
            'detected_valid_corepoint_count': detected_change['valid_corepoint_count'],
            'reference_valid_corepoint_count': reference_change['valid_corepoint_count'],
            'detected_valid_corepoint_percentage': detected_change['valid_corepoint_percentage'],
            'reference_valid_corepoint_percentage': reference_change['valid_corepoint_percentage'],
        })

    n_detected = len(detected_objects)
    n_reference = len(reference_objects)
    n_matched = len(matched_rows)
    object_precision = n_matched / n_detected if n_detected else np.nan
    object_recall = n_matched / n_reference if n_reference else np.nan
    object_f1 = _harmonic_mean(object_precision, object_recall)

    summary = {
        'method': method,
        'n_detected_objects': n_detected,
        'n_reference_objects': n_reference,
        'n_matched_pairs': n_matched,
        'n_unmatched_detected_objects': n_detected - n_matched,
        'n_unmatched_reference_objects': n_reference - n_matched,
        'object_precision': object_precision,
        'object_recall': object_recall,
        'object_f1': object_f1,
    }
    summary.update(_global_support_metrics(detected_objects, reference_objects, detected_time_days, reference_time_days))

    for column in ['spatial_iou', 'temporal_iou', 'spatiotemporal_iou']:
        summary.update(_summary_stats([row[column] for row in matched_rows], column))

    for column in ['change_volume_difference', 'absolute_change_volume_difference']:
        stats = _summary_stats([row[column] for row in matched_rows], column)
        summary[f'mean_{column}'] = stats[f'mean_{column}']
        summary[f'median_{column}'] = stats[f'median_{column}']

    return summary, matched_rows, pair_rows


class _FakeObject:
    def __init__(self, indices, start_epoch, end_epoch):
        self.indices = indices
        self.start_epoch = start_epoch
        self.end_epoch = end_epoch


def _assert_close(actual, expected, label):
    if not np.isclose(actual, expected, equal_nan=True):
        raise AssertionError(f"{label}: expected {expected}, got {actual}")


def _run_evaluation_self_tests():
    irregular_time_days = np.array([0.0, 1.0, 3.0, 6.0])

    metrics = _pairwise_object_metrics(
        _FakeObject([1, 2], 0, 2),
        _FakeObject([1, 2], 0, 2),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['spatial_iou'], 1.0, 'identical spatial IoU')
    _assert_close(metrics['temporal_iou'], 1.0, 'identical temporal IoU')
    _assert_close(metrics['spatiotemporal_iou'], 1.0, 'identical spatiotemporal IoU')

    metrics = _pairwise_object_metrics(
        _FakeObject([1, 2], 0, 2),
        _FakeObject([1, 2], 1, 3),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['spatial_iou'], 1.0, 'same spatial support')
    _assert_close(metrics['spatiotemporal_iou'], metrics['temporal_iou'], 'same spatial support ST IoU')

    metrics = _pairwise_object_metrics(
        _FakeObject([1, 2], 0, 2),
        _FakeObject([2, 3], 0, 2),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['temporal_iou'], 1.0, 'same temporal interval')
    _assert_close(metrics['spatiotemporal_iou'], metrics['spatial_iou'], 'same temporal interval ST IoU')

    metrics = _pairwise_object_metrics(
        _FakeObject([1, 2], 0, 2),
        _FakeObject([2, 3], 1, 3),
        irregular_time_days,
        irregular_time_days,
    )
    if not (
        metrics['spatiotemporal_iou'] <= metrics['spatial_iou'] + _IOU_TOL
        and metrics['spatiotemporal_iou'] <= metrics['temporal_iou'] + _IOU_TOL
    ):
        raise AssertionError('partial overlap ST IoU bound failed')
    if np.isclose(
        metrics['spatiotemporal_iou'],
        metrics['spatial_iou'] * metrics['temporal_iou'],
    ):
        raise AssertionError('ST IoU should not be calculated as spatial_iou * temporal_iou')

    metrics = _pairwise_object_metrics(
        _FakeObject([1, 2], 0, 1),
        _FakeObject([1, 2], 2, 3),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['temporal_iou'], 0.0, 'no temporal overlap')
    _assert_close(metrics['spatiotemporal_iou'], 0.0, 'no temporal overlap ST IoU')

    metrics = _pairwise_object_metrics(
        _FakeObject([1], 0, 2),
        _FakeObject([2], 0, 2),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['spatial_iou'], 0.0, 'no spatial overlap')
    _assert_close(metrics['spatiotemporal_iou'], 0.0, 'no spatial overlap ST IoU')

    pair_rows = _pairwise_metrics_table(
        'test',
        [_FakeObject([1, 2], 0, 2), _FakeObject([1, 2], 1, 3)],
        [_FakeObject([1, 2], 0, 2)],
        irregular_time_days,
        irregular_time_days,
    )
    matrix = _spatiotemporal_iou_matrix(pair_rows, 2, 1)
    matches = _one_to_one_matches(matrix, 0.0)
    if len(matches) != 1 or matches[0] != (0, 0):
        raise AssertionError('one-to-one assignment competition failed')

    metrics = _pairwise_object_metrics(
        _FakeObject([1], 0, 1),
        _FakeObject([1], 0, 2),
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(metrics['temporal_iou'], 1.0 / 3.0, 'irregular timestamp temporal IoU')

    global_metrics = _global_support_metrics(
        [_FakeObject([1], 0, 2), _FakeObject([1], 1, 3)],
        [_FakeObject([1], 0, 3)],
        irregular_time_days,
        irregular_time_days,
    )
    _assert_close(global_metrics['global_tp'], 6.0, 'merged overlapping global TP')
    _assert_close(global_metrics['global_fp'], 0.0, 'merged overlapping global FP')
    _assert_close(global_metrics['global_fn'], 0.0, 'merged overlapping global FN')

    distances = np.array([
        [0.0, 1.0, np.nan],
        [np.nan, 2.0, 3.0],
        [0.0, np.nan, 3.0],
    ])
    change = _mean_time_integrated_change(
        _FakeObject([0, 1, 2], 0, 2),
        distances,
        irregular_time_days,
    )
    if change['valid_corepoint_count'] != 2:
        raise AssertionError('NaN handling should exclude only invalid corepoints')
    _assert_close(change['valid_corepoint_percentage'], 100.0 * 2 / 3, 'valid corepoint percentage')


_run_evaluation_self_tests()
print('Continuous 4D-OBC evaluation self-tests passed.')

_validate_common_corepoints(analyses)
timestamp_info = {name: _timestamps_for_analysis(st_analysis) for name, st_analysis in analyses.items()}
base_timestamps = timestamp_info['GT'][0]
for name, (timestamps_for_method, _) in timestamp_info.items():
    if len(timestamps_for_method) != len(base_timestamps) or any(
        ts != base_ts for ts, base_ts in zip(timestamps_for_method, base_timestamps)
    ):
        raise ValueError(f"{name} timestamps differ from GT timestamps")

print(
    f"Evaluation time axis: {len(base_timestamps)} acquisitions, "
    f"{timestamp_info['GT'][1][0]:.3f} to {timestamp_info['GT'][1][-1]:.3f} elapsed days"
)

gt_analysis = analyses['GT']
summary_rows = []
matched_object_rows = []
pairwise_metric_rows = []
for method in ['M3C2', 'TAM3C2']:
    summary, method_matches, method_pairs = _evaluate_method(
        method,
        analyses[method],
        gt_analysis,
        timestamp_info,
    )
    summary_rows.append(summary)
    matched_object_rows.extend(method_matches)
    pairwise_metric_rows.extend(method_pairs)

if pd is not None:
    summary_df = pd.DataFrame(summary_rows).set_index('method')
    matched_objects_df = pd.DataFrame(matched_object_rows)
    pairwise_metrics_df = pd.DataFrame(pairwise_metric_rows)

    summary_columns = [
        'n_detected_objects',
        'n_reference_objects',
        'n_matched_pairs',
        'n_unmatched_detected_objects',
        'n_unmatched_reference_objects',
        'object_precision',
        'object_recall',
        'object_f1',
        'global_tp',
        'global_fp',
        'global_fn',
        'global_precision',
        'global_recall',
        'global_f1',
        'global_iou',
        'mean_spatial_iou',
        'median_spatial_iou',
        'std_spatial_iou',
        'min_spatial_iou',
        'max_spatial_iou',
        'mean_temporal_iou',
        'median_temporal_iou',
        'std_temporal_iou',
        'min_temporal_iou',
        'max_temporal_iou',
        'mean_spatiotemporal_iou',
        'median_spatiotemporal_iou',
        'std_spatiotemporal_iou',
        'min_spatiotemporal_iou',
        'max_spatiotemporal_iou',
        'mean_change_volume_difference',
        'median_change_volume_difference',
        'mean_absolute_change_volume_difference',
        'median_absolute_change_volume_difference',
    ]
    display(summary_df[summary_columns])

    if len(matched_objects_df) > 0:
        display(
            matched_objects_df.sort_values(
                ['method', 'spatiotemporal_iou'],
                ascending=[True, False],
            )
        )
    else:
        print('No successful one-to-one object matches at the configured threshold.')
else:
    summary_df = summary_rows
    matched_objects_df = matched_object_rows
    pairwise_metrics_df = pairwise_metric_rows
    print('Summary:')
    for row in summary_rows:
        print(row)
    print('Successful one-to-one matches:')
    for row in matched_object_rows:
        print(row)


In [ ]:
# Visual summary of the continuous 4D-OBC comparison.

if pd is not None:
    global_metrics_to_plot = ['global_precision', 'global_recall', 'global_f1', 'global_iou']
    ax = (
        summary_df[global_metrics_to_plot]
        .rename(columns={
            'global_precision': 'Precision',
            'global_recall': 'Recall',
            'global_f1': 'F1',
            'global_iou': 'IoU',
        })
        .plot(kind='bar', figsize=(9, 4), ylim=(0, 1), rot=0)
    )
    ax.set_ylabel('Score')
    ax.set_title('Global continuous-support agreement against GT')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    object_metrics_to_plot = ['object_precision', 'object_recall', 'object_f1']
    ax = (
        summary_df[object_metrics_to_plot]
        .rename(columns={
            'object_precision': 'Object precision',
            'object_recall': 'Object recall',
            'object_f1': 'Object F1',
        })
        .plot(kind='bar', figsize=(9, 4), ylim=(0, 1), rot=0)
    )
    ax.set_ylabel('Score')
    ax.set_title('One-to-one object detection metrics')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    if len(matched_objects_df) > 0 and 'method' in matched_objects_df.columns:
        fig, ax = plt.subplots(figsize=(9, 4))
        for method, group in matched_objects_df.groupby('method'):
            vals = np.sort(group['spatiotemporal_iou'].to_numpy())[::-1]
            ax.plot(np.arange(1, len(vals) + 1), vals, marker='o', ms=3, lw=1, label=method)
        ax.axhline(match_iou_threshold, color='black', ls='--', lw=1, label=f'match threshold={match_iou_threshold}')
        ax.set_xlabel('Successful one-to-one match rank by spatiotemporal IoU')
        ax.set_ylabel('Spatiotemporal IoU')
        ax.set_title('Matched-object quality')
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('No successful one-to-one object matches to plot.')
else:
    print('Install pandas to enable table-based plotting in this cell.')

# old comparison

In [ ]:
# The archives were loaded in Section 2. Keep the shared dictionary interface
# used by the evaluation code below.
analyses = {
    'GT': ref_analysis,
    'M3C2': m3c2_analysis,
    'TAM3C2': analysis,
}

for name, st_analysis in analyses.items():
    objects_for_method = st_analysis.objects
    if objects_for_method is None:
        raise RuntimeError(
            f"{name} analysis has no stored 4D-OBC objects. "
            "Run RegionGrowingAlgorithm in the producer notebook first."
        )
    print(
        f"{name}: {len(objects_for_method)} objects, "
        f"{st_analysis.corepoints.cloud.shape[0]:,} corepoints, "
        f"distance shape={st_analysis.distances.shape}, "
        f"path={st_analysis.filename}"
    )


In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None


def _corepoint_index_mapper(source_analysis, target_analysis):
    source_cp = np.asarray(source_analysis.corepoints.cloud)
    target_cp = np.asarray(target_analysis.corepoints.cloud)

    if source_cp.shape != target_cp.shape or not np.allclose(source_cp, target_cp):
        raise ValueError(
            "All compared archives must use the same corepoint array and ordering."
        )
    return np.arange(len(source_cp)), np.ones(len(source_cp), dtype=bool)


def _object_corepoint_set(obj, index_map=None, valid_source=None):
    idx = np.asarray(obj.indices, dtype=int)
    if index_map is None:
        return set(idx.tolist())

    keep = valid_source[idx]
    return set(index_map[idx[keep]].tolist())


def _object_epoch_set(obj):
    return set(range(int(obj.start_epoch), int(obj.end_epoch) + 1))


def _object_token_set(obj, index_map=None, valid_source=None):
    cp_set = _object_corepoint_set(obj, index_map=index_map, valid_source=valid_source)
    epoch_set = _object_epoch_set(obj)
    return {(cp_idx, epoch_idx) for cp_idx in cp_set for epoch_idx in epoch_set}


def _analysis_token_set(objects_for_method, index_map=None, valid_source=None):
    tokens = set()
    for obj in objects_for_method:
        tokens |= _object_token_set(obj, index_map=index_map, valid_source=valid_source)
    return tokens


def _set_metrics(pred_tokens, gt_tokens):
    tp = len(pred_tokens & gt_tokens)
    fp = len(pred_tokens - gt_tokens)
    fn = len(gt_tokens - pred_tokens)
    union = len(pred_tokens | gt_tokens)
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else np.nan
    iou = tp / union if union else np.nan
    return {
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'iou': iou,
    }


def _iou(a, b):
    union = len(a | b)
    return len(a & b) / union if union else np.nan


def _match_objects(pred_objects, gt_objects, index_map=None, valid_source=None):
    gt_cp_sets = [_object_corepoint_set(obj) for obj in gt_objects]
    gt_epoch_sets = [_object_epoch_set(obj) for obj in gt_objects]
    gt_token_sets = [_object_token_set(obj) for obj in gt_objects]

    rows = []
    for pred_idx, pred_obj in enumerate(pred_objects):
        pred_cp = _object_corepoint_set(pred_obj, index_map=index_map, valid_source=valid_source)
        pred_epochs = _object_epoch_set(pred_obj)
        pred_tokens = _object_token_set(pred_obj, index_map=index_map, valid_source=valid_source)

        best = None
        for gt_idx, (gt_cp, gt_epochs, gt_tokens) in enumerate(zip(gt_cp_sets, gt_epoch_sets, gt_token_sets)):
            st_iou = _iou(pred_tokens, gt_tokens)
            row = {
                'pred_object': pred_idx,
                'gt_object': gt_idx,
                'spatial_iou': _iou(pred_cp, gt_cp),
                'temporal_iou': _iou(pred_epochs, gt_epochs),
                'spacetime_iou': st_iou,
                'pred_corepoints': len(pred_cp),
                'gt_corepoints': len(gt_cp),
                'pred_epochs': len(pred_epochs),
                'gt_epochs': len(gt_epochs),
            }
            if best is None or st_iou > best['spacetime_iou']:
                best = row

        if best is not None:
            best['matched'] = best['spacetime_iou'] >= match_iou_threshold
            rows.append(best)

    return rows


gt_analysis = analyses['GT']
gt_objects = list(gt_analysis.objects)
gt_tokens = _analysis_token_set(gt_objects)

summary_rows = []
match_rows = []
for method in ['M3C2', 'TAM3C2']:
    method_analysis = analyses[method]
    method_objects = list(method_analysis.objects)
    index_map, valid_source = _corepoint_index_mapper(method_analysis, gt_analysis)

    method_tokens = _analysis_token_set(
        method_objects,
        index_map=index_map,
        valid_source=valid_source,
    )
    metrics = _set_metrics(method_tokens, gt_tokens)
    metrics.update({
        'method': method,
        'n_objects': len(method_objects),
        'n_gt_objects': len(gt_objects),
        'matched_objects': 0,
    })

    rows = _match_objects(
        method_objects,
        gt_objects,
        index_map=index_map,
        valid_source=valid_source,
    )
    for row in rows:
        row['method'] = method
    metrics['matched_objects'] = sum(row['matched'] for row in rows)

    summary_rows.append(metrics)
    match_rows.extend(rows)

if pd is not None:
    summary_df = pd.DataFrame(summary_rows).set_index('method')
    matches_df = pd.DataFrame(match_rows)
    display(summary_df[['n_objects', 'n_gt_objects', 'matched_objects', 'precision', 'recall', 'f1', 'iou', 'tp', 'fp', 'fn']])
    if len(matches_df) > 0:
        display(matches_df.sort_values(['method', 'spacetime_iou'], ascending=[True, False]))
    else:
        print('No object-level matches to display because no predicted objects or no GT objects were available.')
else:
    summary_df = summary_rows
    matches_df = match_rows
    print('Summary:')
    for row in summary_rows:
        print(row)
    print('Top matches:')
    for row in sorted(match_rows, key=lambda r: r['spacetime_iou'], reverse=True)[:20]:
        print(row)


In [ ]:

if pd is not None:
    metrics_to_plot = ['precision', 'recall', 'f1', 'iou']
    ax = summary_df[metrics_to_plot].plot(kind='bar', figsize=(9, 4), ylim=(0, 1), rot=0)
    ax.set_ylabel('Score')
    ax.set_title('Global 4D-OBC agreement against GT')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    if len(matches_df) > 0 and 'method' in matches_df.columns:
        fig, ax = plt.subplots(figsize=(9, 4))
        for method, group in matches_df.groupby('method'):
            vals = np.sort(group['spacetime_iou'].to_numpy())[::-1]
            ax.plot(np.arange(1, len(vals) + 1), vals, marker='o', ms=3, lw=1, label=method)
        ax.axhline(match_iou_threshold, color='black', ls='--', lw=1, label=f'match threshold={match_iou_threshold}')
        ax.set_xlabel('Predicted object rank by best GT IoU')
        ax.set_ylabel('Best GT spacetime IoU')
        ax.set_title('Object-level best-match quality')
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('No object-level matches to plot.')
else:
    print('Install pandas to enable table-based plotting in this cell.')

# individual object investigation

In [ ]:
# Investigation: plot all 4D-OBC object delineations for GT, M3C2, and TAM3C2.
# Object delineation is shown as the convex hull of member corepoints.

from scipy.spatial import ConvexHull, QhullError
from matplotlib.patches import Polygon
from matplotlib.lines import Line2D

def _objects_for_analysis(st_analysis):
    objects_for_method = st_analysis.objects
    if objects_for_method is None:
        raise RuntimeError(f"{st_analysis.filename} has no stored 4D-OBC objects.")
    return list(objects_for_method)


def _object_indices(obj):
    return np.asarray(obj.indices, dtype=int)


def _object_xy(st_analysis, obj):
    corepoints_for_method = np.asarray(st_analysis.corepoints.cloud)
    return corepoints_for_method[_object_indices(obj), :2]


def _draw_object_outline(
    ax,
    xy,
    color,
    label=None,
    linewidth=1.4,
    point_size=5,
    point_alpha=0.25,
    fill_alpha=0.06,
):
    xy = np.asarray(xy, dtype=float)
    if len(xy) == 0:
        return None

    xy_unique = np.unique(xy, axis=0)
    ax.scatter(
        xy[:, 0],
        xy[:, 1],
        s=point_size,
        color=color,
        alpha=point_alpha,
        linewidths=0,
    )

    if len(xy_unique) >= 3:
        try:
            hull = ConvexHull(xy_unique)
            hull_xy = xy_unique[hull.vertices]
            patch = Polygon(
                hull_xy,
                closed=True,
                facecolor=color,
                edgecolor=color,
                linewidth=linewidth,
                alpha=fill_alpha,
            )
            ax.add_patch(patch)
            ax.plot(
                np.r_[hull_xy[:, 0], hull_xy[0, 0]],
                np.r_[hull_xy[:, 1], hull_xy[0, 1]],
                color=color,
                linewidth=linewidth,
                alpha=0.95,
                label=label,
            )
            return np.mean(hull_xy, axis=0)
        except QhullError:
            pass

    if len(xy_unique) >= 2:
        order = np.lexsort((xy_unique[:, 1], xy_unique[:, 0]))
        xy_line = xy_unique[order]
        ax.plot(
            xy_line[:, 0],
            xy_line[:, 1],
            color=color,
            linewidth=linewidth,
            alpha=0.95,
            label=label,
        )
    else:
        ax.scatter(
            xy_unique[:, 0],
            xy_unique[:, 1],
            s=30,
            color=color,
            marker="o",
            label=label,
        )

    return np.mean(xy_unique, axis=0)


def plot_all_object_delineations(
    analyses,
    method_names=("GT", "M3C2", "TAM3C2"),
    label_object_ids=True,
):
    fig, axs = plt.subplots(
        1,
        len(method_names),
        figsize=(6.2 * len(method_names), 6),
        constrained_layout=True,
    )
    if len(method_names) == 1:
        axs = [axs]

    for ax, name in zip(axs, method_names):
        st_analysis = analyses[name]
        corepoints_for_method = np.asarray(st_analysis.corepoints.cloud)
        objects_for_method = _objects_for_analysis(st_analysis)

        ax.scatter(
            corepoints_for_method[:, 0],
            corepoints_for_method[:, 1],
            s=0.3,
            color="0.88",
            linewidths=0,
            label="corepoints",
        )

        cmap = plt.colormaps.get_cmap("tab20")
        for object_id, obj in enumerate(objects_for_method):
            color = cmap(object_id % 20)
            xy = _object_xy(st_analysis, obj)
            centroid = _draw_object_outline(ax, xy, color=color)

            if label_object_ids and centroid is not None:
                ax.text(
                    centroid[0],
                    centroid[1],
                    str(object_id),
                    fontsize=8,
                    ha="center",
                    va="center",
                    color="black",
                    bbox=dict(facecolor="white", edgecolor="none", alpha=0.65, pad=1.0),
                )

        ax.set_title(f"{name}: all 4D-OBC object delineations\nn={len(objects_for_method)}")
        ax.set_xlabel("X [m]")
        ax.set_ylabel("Y [m]")
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.2)

    plt.show()


plot_all_object_delineations(analyses)

In [ ]:
reference_object_id = 0  

In [ ]:
# Investigation: inspect one GT/reference object and its best M3C2/TAM3C2 matches.
# Best match here means highest pairwise spatiotemporal IoU against the selected GT object.
# This diagnostic matching is per-reference-object and is not constrained by the global one-to-one assignment.

try:
    import pandas as pd
except ImportError:
    pd = None

use_smoothed_time_series = False

def _distance_series_for_object_plot(st_analysis, use_smoothed=False):
    if use_smoothed and st_analysis.smoothed_distances is not None:
        return np.asarray(st_analysis.smoothed_distances, dtype=float), "smoothed_distances"
    return np.asarray(st_analysis.distances, dtype=float), "distances"


def _timestamp_info_for(name):
    if "timestamp_info" in globals() and name in timestamp_info:
        return timestamp_info[name]
    return _timestamps_for_analysis(analyses[name])


def _best_detected_match_for_reference(reference_object_id, method):
    gt_objects = _objects_for_analysis(analyses["GT"])
    detected_objects = _objects_for_analysis(analyses[method])

    if reference_object_id < 0 or reference_object_id >= len(gt_objects):
        raise IndexError(
            f"reference_object_id={reference_object_id} outside GT object range "
            f"[0, {len(gt_objects) - 1}]"
        )

    ref_obj = gt_objects[reference_object_id]
    _, ref_time_days = _timestamp_info_for("GT")
    _, det_time_days = _timestamp_info_for(method)

    rows = []
    for detected_object_id, det_obj in enumerate(detected_objects):
        metrics = _pairwise_object_metrics(
            det_obj,
            ref_obj,
            det_time_days,
            ref_time_days,
        )
        metrics.update(
            {
                "method": method,
                "reference_object_id": reference_object_id,
                "detected_object_id": detected_object_id,
            }
        )
        rows.append(metrics)

    finite_rows = [
        row for row in rows
        if np.isfinite(row["spatiotemporal_iou"])
    ]
    if not finite_rows:
        return None, rows

    best = max(finite_rows, key=lambda row: row["spatiotemporal_iou"])
    return best, rows


def _single_object_change_summary(name, object_id):
    st_analysis = analyses[name]
    objects_for_method = _objects_for_analysis(st_analysis)
    _, time_days = _timestamp_info_for(name)
    distances = np.asarray(st_analysis.distances, dtype=float)

    return _mean_time_integrated_change(
        objects_for_method[object_id],
        distances,
        time_days,
    )


def _safe_percent_ratio(numerator, denominator):
    if not np.isfinite(numerator) or not np.isfinite(denominator) or denominator == 0:
        return np.nan
    return 100.0 * numerator / denominator


def build_single_reference_match_table(reference_object_id):
    gt_change = _single_object_change_summary("GT", reference_object_id)
    reference_change_volume = gt_change["change_volume"]

    rows = []
    best_matches = {}

    for method in ["M3C2", "TAM3C2"]:
        best, pair_rows = _best_detected_match_for_reference(reference_object_id, method)
        best_matches[method] = best

        if best is None:
            rows.append(
                {
                    "method": method,
                    "reference_object_id": reference_object_id,
                    "best_detected_object_id": None,
                    "spatial_iou_%": np.nan,
                    "temporal_iou_%": np.nan,
                    "spatiotemporal_iou_%": np.nan,
                    "reference_change_volume": reference_change_volume,
                    "detected_change_volume": np.nan,
                    "detected/reference_change_%": np.nan,
                    "change_difference": np.nan,
                    "change_difference_%of_ref": np.nan,
                    "shared_corepoints": np.nan,
                    "reference_corepoints": np.nan,
                    "detected_corepoints": np.nan,
                    "overlapping_duration_days": np.nan,
                    "reference_duration_days": np.nan,
                    "detected_duration_days": np.nan,
                }
            )
            continue

        detected_object_id = int(best["detected_object_id"])
        det_change = _single_object_change_summary(method, detected_object_id)
        detected_change_volume = det_change["change_volume"]
        change_difference = detected_change_volume - reference_change_volume

        rows.append(
            {
                "method": method,
                "reference_object_id": reference_object_id,
                "best_detected_object_id": detected_object_id,
                "spatial_iou_%": 100.0 * best["spatial_iou"],
                "temporal_iou_%": 100.0 * best["temporal_iou"],
                "spatiotemporal_iou_%": 100.0 * best["spatiotemporal_iou"],
                "reference_change_volume": reference_change_volume,
                "detected_change_volume": detected_change_volume,
                "detected/reference_change_%": _safe_percent_ratio(
                    detected_change_volume,
                    reference_change_volume,
                ),
                "change_difference": change_difference,
                "change_difference_%of_ref": _safe_percent_ratio(
                    change_difference,
                    reference_change_volume,
                ),
                "shared_corepoints": best["shared_corepoint_count"],
                "reference_corepoints": best["reference_corepoint_count"],
                "detected_corepoints": best["detected_corepoint_count"],
                "overlapping_duration_days": best["overlapping_duration"],
                "reference_duration_days": best["reference_duration"],
                "detected_duration_days": best["detected_duration"],
            }
        )

    if pd is not None:
        table = pd.DataFrame(rows)
    else:
        table = rows

    return table, best_matches


single_match_table, best_matches = build_single_reference_match_table(reference_object_id)

print(f"Selected GT/reference object id: {reference_object_id}")
for method, best in best_matches.items():
    if best is None:
        print(f"{method}: no finite best match")
    else:
        print(
            f"{method}: best object id={int(best['detected_object_id'])}, "
            f"ST-IoU={100.0 * best['spatiotemporal_iou']:.2f}%, "
            f"spatial IoU={100.0 * best['spatial_iou']:.2f}%, "
            f"temporal IoU={100.0 * best['temporal_iou']:.2f}%"
        )

if pd is not None:
    display(single_match_table)
else:
    for row in single_match_table:
        print(row)


def _selected_object_ids(reference_object_id, best_matches):
    selected = {"GT": reference_object_id}
    for method in ["M3C2", "TAM3C2"]:
        best = best_matches.get(method)
        selected[method] = None if best is None else int(best["detected_object_id"])
    return selected


selected_object_ids = _selected_object_ids(reference_object_id, best_matches)


def plot_single_reference_spatial_comparison(selected_object_ids):
    colors = {
        "GT": "black",
        "M3C2": "tab:blue",
        "TAM3C2": "tab:orange",
    }

    fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)

    gt_corepoints = np.asarray(analyses["GT"].corepoints.cloud)
    ax.scatter(
        gt_corepoints[:, 0],
        gt_corepoints[:, 1],
        s=0.4,
        color="0.88",
        linewidths=0,
        label="all corepoints",
    )

    legend_handles = []
    for name in ["GT", "M3C2", "TAM3C2"]:
        object_id = selected_object_ids[name]
        if object_id is None:
            continue

        st_analysis = analyses[name]
        obj = _objects_for_analysis(st_analysis)[object_id]
        xy = _object_xy(st_analysis, obj)
        color = colors[name]

        centroid = _draw_object_outline(
            ax,
            xy,
            color=color,
            label=f"{name} object {object_id}",
            linewidth=2.2 if name == "GT" else 1.8,
            point_size=14 if name == "GT" else 10,
            point_alpha=0.55,
            fill_alpha=0.08,
        )

        ax.scatter(
            xy[:, 0],
            xy[:, 1],
            s=16 if name == "GT" else 11,
            color=color,
            alpha=0.55,
            linewidths=0,
        )

        if centroid is not None:
            ax.text(
                centroid[0],
                centroid[1],
                f"{name}\nID {object_id}",
                ha="center",
                va="center",
                fontsize=9,
                bbox=dict(facecolor="white", edgecolor=color, alpha=0.8, pad=2),
            )

        legend_handles.append(
            Line2D([0], [0], color=color, lw=2, label=f"{name} object {object_id}")
        )

    ax.set_title(
        "Spatial delineation of selected GT object and best M3C2/TAM3C2 matches"
    )
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.25)
    ax.legend(handles=legend_handles, loc="best")
    plt.show()


plot_single_reference_spatial_comparison(selected_object_ids)


def _object_interval_timestamps(name, object_id):
    st_analysis = analyses[name]
    obj = _objects_for_analysis(st_analysis)[object_id]
    timestamps, time_days = _timestamp_info_for(name)
    props = _object_properties(obj, time_days)

    return {
        "start_timestamp": timestamps[props["start_epoch"]],
        "end_timestamp": timestamps[props["end_epoch"]],
        "start_time_days": props["start_time"],
        "end_time_days": props["end_time"],
        "duration_days": props["duration"],
    }


def plot_single_reference_temporal_overlap(selected_object_ids):
    colors = {
        "GT": "black",
        "M3C2": "tab:blue",
        "TAM3C2": "tab:orange",
    }

    intervals = {}
    for name, object_id in selected_object_ids.items():
        if object_id is not None:
            intervals[name] = _object_interval_timestamps(name, object_id)

    fig, ax = plt.subplots(figsize=(11, 2.8), constrained_layout=True)

    y_positions = {
        "GT": 2,
        "M3C2": 1,
        "TAM3C2": 0,
    }

    for name in ["GT", "M3C2", "TAM3C2"]:
        if name not in intervals:
            continue

        interval = intervals[name]
        y = y_positions[name]
        color = colors[name]

        ax.hlines(
            y,
            interval["start_timestamp"],
            interval["end_timestamp"],
            color=color,
            linewidth=8,
            alpha=0.75,
            label=f"{name} object {selected_object_ids[name]}",
        )
        ax.scatter(
            [interval["start_timestamp"], interval["end_timestamp"]],
            [y, y],
            color=color,
            s=45,
            zorder=3,
        )

    if len(intervals) >= 2:
        overlap_start = max(interval["start_timestamp"] for interval in intervals.values())
        overlap_end = min(interval["end_timestamp"] for interval in intervals.values())
        if overlap_start <= overlap_end:
            ax.axvspan(
                overlap_start,
                overlap_end,
                color="green",
                alpha=0.15,
                label="common overlap",
            )

    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(["TAM3C2", "M3C2", "GT"])
    ax.set_title("Temporal intervals of selected objects")
    ax.set_xlabel("Date")
    ax.grid(axis="x", alpha=0.25)
    ax.legend(loc="upper right")
    plt.show()


plot_single_reference_temporal_overlap(selected_object_ids)


def plot_single_reference_object_time_series(selected_object_ids):
    colors = {
        "GT": "black",
        "M3C2": "tab:blue",
        "TAM3C2": "tab:orange",
    }

    fig, axs = plt.subplots(
        3,
        1,
        figsize=(12, 9),
        sharex=True,
        constrained_layout=True,
    )

    for ax, name in zip(axs, ["GT", "M3C2", "TAM3C2"]):
        object_id = selected_object_ids[name]
        color = colors[name]

        if object_id is None:
            ax.text(
                0.5,
                0.5,
                f"{name}: no matched object",
                transform=ax.transAxes,
                ha="center",
                va="center",
            )
            ax.set_axis_off()
            continue

        st_analysis = analyses[name]
        objects_for_method = _objects_for_analysis(st_analysis)
        obj = objects_for_method[object_id]
        indices = _object_indices(obj)

        timestamps, time_days = _timestamp_info_for(name)
        distance_series, distance_label = _distance_series_for_object_plot(
            st_analysis,
            use_smoothed=use_smoothed_time_series,
        )

        object_series = distance_series[indices, :]

        for series in object_series:
            ax.plot(
                timestamps,
                series,
                color=color,
                alpha=0.12,
                linewidth=0.7,
            )

        with np.errstate(all="ignore"):
            mean_series = np.nanmean(object_series, axis=0)

        ax.plot(
            timestamps,
            mean_series,
            color="black",
            linewidth=2.0,
            label="mean object corepoint timeseries",
        )

        interval = _object_interval_timestamps(name, object_id)
        ax.axvspan(
            interval["start_timestamp"],
            interval["end_timestamp"],
            color=color,
            alpha=0.16,
            label="object timespan",
        )

        ax.axhline(0, color="0.5", linewidth=0.8, linestyle="--")
        ax.set_ylabel("Distance [m]")
        ax.set_title(
            f"{name} object {object_id}: "
            f"{len(indices)} corepoints, source={distance_label}"
        )
        ax.grid(alpha=0.25)
        ax.legend(loc="best")

    axs[-1].set_xlabel("Date")
    plt.show()


plot_single_reference_object_time_series(selected_object_ids)